In [2]:
from sentence_transformers import SentenceTransformer

# 1. Load a model that supports both text and images
model = SentenceTransformer("Qwen/Qwen3-VL-Embedding-2B")

# 2. Encode images from URLs
img_embeddings = model.encode([
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg",
])

# 3. Encode text queries (one matching + one hard negative per image)
text_embeddings = model.encode([
    "A green car parked in front of a yellow building",
    "A red car driving on a highway",
    "A bee on a pink flower",
    "A wasp on a wooden table",
])

# 4. Compute cross-modal similarities
similarities = model.similarity(text_embeddings, img_embeddings)
print(similarities)
# tensor([[0.5115, 0.1078],
#         [0.1999, 0.1108],
#         [0.1255, 0.6749],
#         [0.1283, 0.2704]])

/home/monster/Desktop/codes/Multimodal-Language-Models/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 625/625 [00:00<00:00, 6283.23it/s]
Default prompt name is set to 'default'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


tensor([[0.5116, 0.1095],
        [0.2030, 0.1115],
        [0.1257, 0.6724],
        [0.1254, 0.2714]])


In [3]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'image': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'video': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'message': {'method': 'forward', 'method_output_name': 'last_hidden_state', 'format': 'structured'}}, 'module_output_name': 'token_embeddings', 'processing_kwargs': {'chat_template': {'add_generation_prompt': True}}, 'unpad_inputs': False, 'architecture': 'Qwen3VLModel'})
  (1): Pooling({'embedding_dimension': 2048, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

### Cross Encoder

Characteristics of Cross Encoder (a.k.a reranker) models calculates a similarity score given pairs of inputs (typically text, but also images or other modalities). They generally provides superior performance compared to a Sentence Transformer (a.k.a. bi-encoder) model. However, they are often slower than a Sentence Transformer model, as it requires computation for each pair rather than each text. 

Use-case: Cross Encoders are often used to re-rank the top-k results from a Sentence Transformer model.


In [4]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("Qwen/Qwen3-VL-Reranker-2B")

query = "A green car parked in front of a yellow building"
documents = [
    # Image documents (URL or local file path)
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg",
    # Text document
    "A vintage Volkswagen Beetle painted in bright green sits in a driveway.",
    # Combined text + image document
    {
        "text": "A car in a European city",
        "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg",
    },
]

rankings = model.rank(query, documents)
for rank in rankings:
    print(f"{rank['score']:.4f}\t(document {rank['corpus_id']})")
"""
0.9375  (document 0)
0.5000  (document 3)
-1.2500 (document 2)
-2.4375 (document 1)
"""

Loading weights: 100%|██████████| 625/625 [00:00<00:00, 2553.61it/s]
Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


0.8750	(document 0)
0.5625	(document 3)
-0.7500	(document 2)
-2.3750	(document 1)


'\n0.9375  (document 0)\n0.5000  (document 3)\n-1.2500 (document 2)\n-2.4375 (document 1)\n'

Let's inspect the cross encoder model

In [5]:
model

CrossEncoder(
  (0): Transformer({'transformer_task': 'any-to-any', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}, 'image': {'method': 'forward', 'method_output_name': 'logits'}, 'video': {'method': 'forward', 'method_output_name': 'logits'}, 'message': {'method': 'forward', 'method_output_name': 'logits', 'format': 'structured'}}, 'module_output_name': 'causal_logits', 'processing_kwargs': {'chat_template': {'chat_template': 'reranker', 'add_generation_prompt': True}}, 'unpad_inputs': False, 'architecture': 'Qwen3VLForConditionalGeneration'})
  (1): LogitScore({'true_token_id': 9693, 'false_token_id': 2152, 'module_input_name': 'causal_logits'})
)

## Sparse Encoder

Sparse Encoder models calculates sparse vector representations where most dimensions are zero. It provides efficiency benefits for large-scale retrieval systems due to the sparse nature of embeddings. They are often more interpretable than dense embeddings, with non-zero dimensions corresponding to specific tokens. They can be complementary to dense embeddings, enabling hybrid search systems that combine the strengths of both approaches.


In [ ]:
from sentence_transformers import SparseEncoder

# 1. Load a pretrained SparseEncoder model
model = SparseEncoder("naver/splade-cocondenser-ensembledistil")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate sparse embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 30522] - sparse representation with vocabulary size dimensions

# 3. Calculate the embedding similarities (using dot product by default)
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[   35.629,     9.154,     0.098],
#         [    9.154,    27.478,     0.019],
#         [    0.098,     0.019,    29.553]])

# 4. Check sparsity statistics
stats = SparseEncoder.sparsity(embeddings)
print(f"Sparsity: {stats['sparsity_ratio']:.2%}")  # Typically >99% zeros
print(f"Avg non-zero dimensions per embedding: {stats['active_dims']:.2f}")

## Multi-Vector Encoder

Multi-Vector Encoder (a.k.a ColBERT-style or “late-interaction”) models calculates a sequence of token-level vectors per input rather than a single fixed-size embedding. Scores queries against documents with the MaxSim operator: for each query token, take the maximum similarity to any document token, then sum across query tokens.

Multi-vector encoder preserves token-level matching information that single-vector models discard, typically yielding stronger retrieval at the cost of a larger index footprint.

Recent VLM-backed variants (ColPali, ColQwen2, ColModernVBert, …) extend this to image documents (each image patch becomes a “token”), enabling end-to-end OCR-free document retrieval.

The usage for Multi-Vector Encoder models follows a similar pattern to Sentence Transformers:

In [ ]:
from sentence_transformers import MultiVectorEncoder

# 1. Load a model that matches text queries against page images, no OCR step
model = MultiVectorEncoder("vidore/colqwen2.5-v0.2")

queries = [
    "What is the variable represented on the y-axis of the graph?",
    "Total outlay is maximum in which year?",
]
# Image documents are passed as URLs, local paths, or PIL images
images = [
    "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/doc1.jpg",
    "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/doc2.jpg",
    "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/doc3.jpg",
    "https://huggingface.co/datasets/sentence-transformers/example-documents/resolve/main/doc4.jpg",
]

# 2. Encode with the same two calls as for text
query_embeddings = model.encode_query(queries)
document_embeddings = model.encode_document(images)

# A page yields far more vectors than a query: one per image patch
print(query_embeddings[0].shape, document_embeddings[0].shape)
# torch.Size([25, 128]) torch.Size([755, 128])

# 3. Score query text tokens against document image patches with MaxSim
scores = model.similarity(query_embeddings, document_embeddings)
print(scores)
# tensor([[13.8672, 12.3115, 12.1670, 11.0293],
#         [ 7.2012, 14.7207,  6.9414,  6.9746]])

Default pooling for CausalLM models

When loading a model without a pre-trained Sentence Transformer configuration (e.g. a raw transformers model), previous versions always defaulted to mean pooling. Starting with v5.4, CausalLM-based models (e.g. Llama, Qwen, Mistral) now default to last token pooling instead, as the last token has the most complete contextual representation in causal models. All other models continue to default to mean pooling.

If you relied on mean pooling for a CausalLM model, you can explicitly set the pooling mode:

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.modules import Transformer, Pooling

transformer = Transformer("Qwen/Qwen2-0.5B")
pooling = Pooling(transformer.get_embedding_dimension(), pooling_mode="mean")
model = SentenceTransformer(modules=[transformer, pooling])

In [6]:
print(model.modalities)

['text', 'image', 'video', 'message']
